# PolarQuant: A Visual & Mathematical Walkthrough

This notebook breaks down the PolarQuant algorithm step by step, with visuals and code.

**Prerequisites**: Basic linear algebra (vectors, dot products) and basic probability (mean, variance).

---

## What Problem Does PolarQuant Solve?

Large Language Models (LLMs) generate text one token at a time.  
To avoid recomputing attention for all past tokens, they store **Key** and **Value** vectors in a **KV cache**.

**Problem**: The KV cache grows with sequence length × model layers × attention heads.  
For a 128K context window, this can consume **tens of GB** of GPU memory.

**PolarQuant's solution**: Convert vectors to **polar coordinates**, then quantize the **angles** instead of the raw numbers.  
After a random rotation trick, those angles have a beautifully predictable distribution —  
so predictable that we don't need to store any scale/zero-point normalization parameters.

This saves **>4.2× memory** with near-zero quality loss.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
from scipy.special import gamma as gamma_fn
from scipy.stats import norm

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12

---
## Step 0: Cartesian vs Polar Coordinates (2D Refresher)

Every 2D point $(x, y)$ can be represented two ways:

| Cartesian | Polar |
|---|---|
| $(x, y)$ | $(r, \theta)$ |
| Two lengths | A length + an angle |

The conversion:

$$r = \sqrt{x^2 + y^2}, \quad \theta = \arctan\left(\frac{y}{x}\right)$$

$$x = r \cos\theta, \quad y = r \sin\theta$$

**Key insight**: If $r$ is known, the entire 2D vector is determined by a single angle $\theta$.  
One angle replaces two coordinates — that's where the compression comes from!

In [ ]:
# 2D polar coordinates demo
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Example point
x, y = 3.0, 4.0
r = np.sqrt(x**2 + y**2)
theta = np.arctan2(y, x)

# Cartesian view
ax = axes[0]
ax.arrow(0, 0, x, y, head_width=0.15, head_length=0.15, fc='steelblue', ec='steelblue', linewidth=2)
ax.plot([x, x], [0, y], 'k--', alpha=0.4)
ax.plot([0, x], [y, y], 'k--', alpha=0.4)
ax.scatter([x], [y], color='red', s=100, zorder=5)
ax.annotate(f'({x:.0f}, {y:.0f})', (x+0.2, y+0.2), fontsize=13, color='red')
ax.annotate(f'x = {x:.0f}', (x/2, -0.5), fontsize=11, ha='center')
ax.annotate(f'y = {y:.0f}', (x+0.3, y/2), fontsize=11)
ax.set_xlim(-0.5, 6)
ax.set_ylim(-1, 6)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_title('Cartesian: (x, y)', fontsize=14)
ax.axhline(0, color='k', linewidth=0.5)
ax.axvline(0, color='k', linewidth=0.5)

# Polar view
ax = axes[1]
ax.arrow(0, 0, x, y, head_width=0.15, head_length=0.15, fc='steelblue', ec='steelblue', linewidth=2)
ax.scatter([x], [y], color='red', s=100, zorder=5)
# Draw angle arc
angles = np.linspace(0, theta, 50)
arc_r = 1.5
ax.plot(arc_r*np.cos(angles), arc_r*np.sin(angles), 'green', linewidth=2)
ax.annotate(f'θ = {np.degrees(theta):.1f}°', (1.8, 0.8), fontsize=13, color='green')
ax.annotate(f'r = {r:.1f}', (x/2-0.3, y/2+0.3), fontsize=13, color='steelblue', rotation=np.degrees(theta))
ax.set_xlim(-0.5, 6)
ax.set_ylim(-1, 6)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_title('Polar: (r, θ)', fontsize=14)
ax.axhline(0, color='k', linewidth=0.5)
ax.axvline(0, color='k', linewidth=0.5)

plt.suptitle('Same point, two representations', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"Cartesian: x={x}, y={y}")
print(f"Polar:     r={r:.2f}, θ={np.degrees(theta):.1f}°")
print(f"\nIf we know r, we only need to store θ to recover (x,y)!")

---
## Step 1: Recursive Polar Transform — Extending to High Dimensions

In 2D, one angle captures a pair of coordinates. But KV vectors have $d=128$ dimensions!  
PolarQuant uses a **recursive** approach:

### Level 1: Pair up coordinates
Group the $d$ coordinates into $d/2$ pairs: $(x_1, x_2), (x_3, x_4), \ldots$  
Convert each pair to polar: get $d/2$ angles + $d/2$ radii.

### Level 2: Pair up the radii
Take the $d/2$ radii, group them into $d/4$ pairs, convert to polar again.  
Get $d/4$ new angles + $d/4$ new radii.

### Repeat...
After $\log_2 d$ levels, you have:
- **1 final radius** ($= \|\mathbf{x}\|$, the vector's length)
- **$d - 1$ angles** across all levels

```
Level 1: d/2 angles from d coordinates        (widest level)
Level 2: d/4 angles from d/2 radii
Level 3: d/8 angles from d/4 radii
  ...      ...        ...
Level log₂d: 1 angle from 2 radii             (top level)
──────────────────────────────────────
Total: d/2 + d/4 + ... + 1 = d - 1 angles
```

The radius is stored in full precision. All angles are quantized.

In [ ]:
def recursive_polar_transform(x):
    """
    Recursively convert a vector to polar coordinates.
    Returns: (radius, list_of_angle_vectors)
    """
    d = len(x)
    all_angles = []
    current = x.copy()
    
    while len(current) > 1:
        n = len(current)
        radii = np.zeros(n // 2)
        angles = np.zeros(n // 2)
        
        for j in range(n // 2):
            a, b = current[2*j], current[2*j + 1]
            radii[j] = np.sqrt(a**2 + b**2)
            angles[j] = np.arctan2(abs(b), abs(a))  # angle in [0, π/2]
            if len(all_angles) == 0:  # level 1: full angle
                angles[j] = np.arctan2(b, a)  # angle in [0, 2π)
        
        all_angles.append(angles)
        current = radii
    
    return current[0], all_angles  # final radius + all angle vectors


def inverse_polar_transform(radius, all_angles):
    """
    Reconstruct vector from polar representation.
    """
    current = np.array([radius])
    
    for angles in reversed(all_angles):
        expanded = np.zeros(len(current) * 2)
        for j in range(len(current)):
            expanded[2*j] = current[j] * np.cos(angles[j])
            expanded[2*j + 1] = current[j] * np.sin(angles[j])
        current = expanded
    
    return current


# Demo: 8-dimensional vector
d = 8
x = np.random.randn(d)
x = x / np.linalg.norm(x)  # normalize

radius, angles_list = recursive_polar_transform(x)

print(f"Original vector (d={d}): {x.round(4)}")
print(f"\nPolar representation:")
print(f"  Radius: {radius:.6f}")
for i, angles in enumerate(angles_list):
    print(f"  Level {i+1} ({len(angles)} angles): {np.degrees(angles).round(2)}°")
print(f"\nTotal: 1 radius + {sum(len(a) for a in angles_list)} angles = {1 + sum(len(a) for a in angles_list)} values for {d} coordinates")

# Verify reconstruction
x_reconstructed = inverse_polar_transform(radius, angles_list)
print(f"\nReconstruction error: {np.linalg.norm(x - x_reconstructed):.2e} (should be ~0)")

In [ ]:
# Visualize the recursive tree structure
fig, ax = plt.subplots(figsize=(14, 6))

d = 16
levels = int(np.log2(d))
colors = plt.cm.Set2(np.linspace(0, 1, levels + 1))

# Draw tree from bottom (coordinates) to top (final radius)
y_positions = {}
for level in range(levels + 1):
    n_nodes = d // (2**level)
    y = level * 1.5
    for i in range(n_nodes):
        x_pos = (i + 0.5) * (14 / n_nodes)
        y_positions[(level, i)] = (x_pos, y)
        
        if level == 0:
            label = f'$x_{{{i+1}}}$'
            ax.text(x_pos, y, label, ha='center', va='center', fontsize=9,
                    bbox=dict(boxstyle='round,pad=0.3', facecolor=colors[0], alpha=0.7))
        elif level < levels:
            ax.text(x_pos, y, f'$r, \\theta$', ha='center', va='center', fontsize=8,
                    bbox=dict(boxstyle='round,pad=0.3', facecolor=colors[level], alpha=0.7))
        else:
            ax.text(x_pos, y, f'$R$', ha='center', va='center', fontsize=11, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.4', facecolor='gold', alpha=0.9))

# Draw connections
for level in range(1, levels + 1):
    n_parent = d // (2**level)
    for i in range(n_parent):
        px, py = y_positions[(level, i)]
        cx1, cy1 = y_positions[(level-1, 2*i)]
        cx2, cy2 = y_positions[(level-1, 2*i+1)]
        ax.plot([cx1, px], [cy1+0.15, py-0.15], '-', color='gray', alpha=0.5)
        ax.plot([cx2, px], [cy2+0.15, py-0.15], '-', color='gray', alpha=0.5)

# Labels on right
labels = ['Coordinates\n(d values)', f'Level 1\n({d//2} angles)', f'Level 2\n({d//4} angles)',
          f'Level 3\n({d//8} angles)', f'Level 4\n(1 radius)']
for level in range(min(levels+1, len(labels))):
    ax.text(14.5, level*1.5, labels[level], fontsize=9, va='center')

ax.set_xlim(-0.5, 17)
ax.set_ylim(-0.5, levels * 1.5 + 0.5)
ax.axis('off')
ax.set_title(f'Recursive Polar Transform (d={d}): pairs of values → (radius, angle)', fontsize=13)
plt.tight_layout()
plt.show()

---
## Step 2: The Problem with Raw Angles — Unpredictable Distributions

If we directly transform raw KV cache vectors to polar coordinates, the angles have **messy, unpredictable distributions** that vary per layer, per head, per input.

Traditional quantization handles this by computing **per-block statistics** (min, max, scale, zero-point) and storing them in full precision (16 bits). This **normalization overhead** can add >1 extra bit per quantized number!

For example, if you group 128 values into a block and store 2 float16 constants (scale + zero-point):  
Overhead = $\frac{2 \times 16}{128} = 0.25$ bits per value — significant when your target is 2-3 bits!

### PolarQuant's solution: Random Preconditioning

Multiply all vectors by a shared random matrix $\mathbf{S}$ **before** the polar transform.
This makes the angle distributions **universal and predictable**, eliminating normalization entirely.

In [ ]:
# Show angle distributions WITH and WITHOUT random preconditioning
d = 128
n_vectors = 2000

# Simulate "real" KV cache vectors (structured, non-random)
# Real KV vectors have outliers and structure
kv_vectors = np.random.randn(n_vectors, d) * np.random.exponential(1, size=(1, d))
kv_vectors[:, 0] *= 10  # simulate outlier channel
kv_vectors[:, 1] *= 8

# Random preconditioning matrix (shared across all vectors)
G = np.random.randn(d, d)
S, _ = np.linalg.qr(G)  # orthogonal random matrix

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for col, level_idx in enumerate([0, 1, 2]):
    level_label = f'Level {level_idx + 1}'
    
    # WITHOUT preconditioning
    angles_no_precond = []
    for v in kv_vectors[:500]:
        _, angle_list = recursive_polar_transform(v)
        if level_idx < len(angle_list):
            angles_no_precond.extend(angle_list[level_idx])
    
    # WITH preconditioning
    angles_precond = []
    for v in kv_vectors[:500]:
        v_precond = S @ v
        _, angle_list = recursive_polar_transform(v_precond)
        if level_idx < len(angle_list):
            angles_precond.extend(angle_list[level_idx])
    
    ax_top = axes[0, col]
    ax_bot = axes[1, col]
    
    ax_top.hist(np.degrees(angles_no_precond), bins=60, density=True, alpha=0.7,
                color='#e74c3c', edgecolor='white')
    ax_top.set_title(f'{level_label} — WITHOUT preconditioning')
    ax_top.set_xlabel('Angle (degrees)')
    
    ax_bot.hist(np.degrees(angles_precond), bins=60, density=True, alpha=0.7,
                color='#2ecc71', edgecolor='white')
    ax_bot.set_title(f'{level_label} — WITH preconditioning')
    ax_bot.set_xlabel('Angle (degrees)')
    
    if level_idx >= 1:
        ax_bot.axvline(45, color='black', linestyle='--', alpha=0.5, label='π/4 = 45°')
        ax_bot.legend(fontsize=9)

plt.suptitle('Random preconditioning makes angle distributions predictable & concentrated', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## Step 3: The Angle Distribution — Why It Works

After random preconditioning, the preconditioned vector $\mathbf{S} \cdot \mathbf{x}$ becomes a **Gaussian random vector** $\sim \mathcal{N}(0, \|\mathbf{x}\|^2 \mathbf{I})$.

When we apply the polar transform, the angles at level $\ell \geq 2$ follow this distribution:

$$f_{\ell}(\psi) = \frac{\Gamma(2^{\ell-1})}{2^{2^{\ell-1}-2} \cdot \Gamma(2^{\ell-2})^2} \cdot \sin^{2^{\ell-1}-1}(2\psi), \quad \psi \in [0, \pi/2]$$

Don't memorize this! Here's what matters:

| Level $\ell$ | Effective dimension | Angle distribution | Variance |
|---|---|---|---|
| 1 | 2 | Uniform on $[0, 2\pi)$ | Flat |
| 2 | 2 | $\sin(2\psi)$ shaped | $O(1/2)$ |
| 3 | 4 | $\sin^3(2\psi)$ shaped | $O(1/4)$ |
| 4 | 8 | $\sin^7(2\psi)$ shaped | $O(1/8)$ |
| $\ell$ | $2^{\ell-1}$ | $\sin^{2^{\ell-1}-1}(2\psi)$ | $O(1/2^{\ell-1})$ |

**The key property**: 
- Mean = $\pi/4$ (always centered at 45°)
- Variance = $O(1/2^{\ell-1})$ → **higher levels are more concentrated**
- Higher levels need **fewer bits** to quantize accurately!

In [ ]:
# Visualize theoretical angle distributions at each level
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

psi = np.linspace(0.001, np.pi/2 - 0.001, 500)

for idx, level in enumerate([1, 2, 3, 4]):
    ax = axes[idx]
    
    if level == 1:
        # Level 1: uniform on [0, 2π)
        psi_full = np.linspace(0, 2*np.pi, 500)
        pdf = np.ones_like(psi_full) / (2*np.pi)
        ax.plot(np.degrees(psi_full), pdf, 'b-', lw=2)
        ax.fill_between(np.degrees(psi_full), pdf, alpha=0.2, color='blue')
        ax.set_xlabel('Angle (degrees)')
        ax.set_title(f'Level 1\nUniform [0°, 360°)\nVar = large')
    else:
        # Level ℓ ≥ 2: sin^(2^(ℓ-1) - 1)(2ψ)
        eff_dim = 2**(level - 1)
        exponent = eff_dim - 1
        
        # Compute PDF (unnormalized, then normalize)
        pdf = np.sin(2 * psi)**exponent
        # Normalize
        pdf = pdf / np.trapz(pdf, psi)
        
        ax.plot(np.degrees(psi), pdf, 'b-', lw=2)
        ax.fill_between(np.degrees(psi), pdf, alpha=0.2, color='blue')
        ax.axvline(45, color='red', linestyle='--', alpha=0.5, label='π/4 = 45°')
        ax.set_xlabel('Angle (degrees)')
        
        variance = 1 / eff_dim  # approximate
        ax.set_title(f'Level {level}\nsin$^{{{exponent}}}$(2ψ)\nVar ≈ {variance:.3f}')
        ax.legend(fontsize=8)

plt.suptitle('Angle distributions get sharper at higher levels → fewer bits needed!', fontsize=13, y=1.03)
plt.tight_layout()
plt.show()

---
## Step 4: Quantizing the Angles — Level-Adaptive Codebooks

Since we know the exact distribution at each level, we compute **optimal centroids** for each level  
using 1D k-means (Lloyd-Max algorithm) — the same approach as TurboQuant, but on angle distributions.

### Key difference from traditional quantization

| Traditional | PolarQuant |
|---|---|
| Quantize raw coordinate values | Quantize angles in polar coords |
| Needs per-block scale + zero-point | **No normalization needed** |
| Must store 2 × float16 per block | Angles have a known, fixed range |
| Overhead: 0.25-1+ bits/value | Overhead: ~0 bits/value |

### Practical bit allocation (from the paper)

The paper uses $L=4$ recursion levels:
- **Level 1**: 4 bits (range is $[0, 2\pi)$ — widest, needs most precision)
- **Levels 2-4**: 2 bits each (range is $[0, \pi/2]$ — narrower, concentrated)
- **Radius**: stored in full precision (16 bits)

For a block of 16 coordinates: $16 + 32 + 8 + 4 + 2 = 62$ bits → **3.875 bits/coordinate**

In [ ]:
def lloyd_max_1d(pdf_func, support, n_levels, n_samples=100000, max_iter=200):
    """
    Lloyd-Max quantizer for a 1D distribution.
    Uses samples from the distribution to run 1D k-means.
    """
    # Generate samples from the PDF via rejection sampling
    x_range = np.linspace(support[0], support[1], 10000)
    pdf_vals = pdf_func(x_range)
    pdf_max = pdf_vals.max()
    
    samples = []
    while len(samples) < n_samples:
        candidates = np.random.uniform(support[0], support[1], n_samples * 2)
        u = np.random.uniform(0, pdf_max, len(candidates))
        accepted = candidates[u < pdf_func(candidates)]
        samples.extend(accepted.tolist())
    samples = np.array(samples[:n_samples])
    
    # Run 1D k-means
    centroids = np.linspace(support[0], support[1], n_levels)
    for _ in range(max_iter):
        # Assign each sample to nearest centroid
        dists = np.abs(samples[:, None] - centroids[None, :])
        assignments = np.argmin(dists, axis=1)
        new_centroids = np.array([samples[assignments == k].mean() 
                                   if np.sum(assignments == k) > 0 else centroids[k]
                                   for k in range(n_levels)])
        if np.allclose(centroids, new_centroids, atol=1e-10):
            break
        centroids = new_centroids
    
    centroids.sort()
    boundaries = np.concatenate([[support[0]], (centroids[:-1] + centroids[1:]) / 2, [support[1]]])
    return centroids, boundaries


# Compute optimal codebooks for each level
def angle_pdf(psi, level):
    """PDF of angles at a given level (unnormalized, we'll normalize in Lloyd-Max)."""
    if level == 1:
        return np.ones_like(psi)  # uniform
    eff_dim = 2**(level - 1)
    return np.sin(2 * psi)**(eff_dim - 1)


fig, axes = plt.subplots(1, 4, figsize=(16, 4))
bits_per_level = [4, 2, 2, 2]

for idx, (level, b) in enumerate(zip([1, 2, 3, 4], bits_per_level)):
    ax = axes[idx]
    n_levels = 2**b
    
    if level == 1:
        support = (0, 2*np.pi)
    else:
        support = (0, np.pi/2)
    
    pdf_fn = lambda psi, l=level: angle_pdf(psi, l)
    centroids, boundaries = lloyd_max_1d(pdf_fn, support, n_levels, n_samples=50000)
    
    # Plot distribution + centroids
    xx = np.linspace(support[0] + 0.001, support[1] - 0.001, 500)
    pdf = pdf_fn(xx)
    pdf = pdf / np.trapz(pdf, xx)  # normalize
    ax.plot(np.degrees(xx), pdf, 'b-', lw=2)
    ax.fill_between(np.degrees(xx), pdf, alpha=0.15, color='blue')
    
    for c in centroids:
        ax.axvline(np.degrees(c), color='red', lw=1.5, alpha=0.7)
    for bd in boundaries[1:-1]:
        ax.axvline(np.degrees(bd), color='gray', lw=0.8, linestyle='--', alpha=0.4)
    
    ax.set_title(f'Level {level}: {b}-bit ({n_levels} centroids)')
    ax.set_xlabel('Angle (degrees)')

plt.suptitle('Optimal codebooks per level — centroids cluster where probability is highest', fontsize=13, y=1.03)
plt.tight_layout()
plt.show()

---
## Step 5: The Complete PolarQuant Algorithm

### Setup (once per model)
1. Generate a random orthogonal matrix $\mathbf{S}$ (shared across all layers/heads)
2. Precompute optimal angle codebooks for each level

### Quantize (per vector)
1. $\mathbf{y} = \mathbf{S} \cdot \mathbf{x}$ — random preconditioning
2. $(R, \psi^{(1)}, \psi^{(2)}, \ldots, \psi^{(L)}) = \text{Polar}(\mathbf{y})$ — recursive polar transform
3. For each angle: $\text{idx} = \text{nearest\_centroid}(\psi)$ — quantize with level-specific codebook
4. Store: indices + radius $R$ in full precision

### Dequantize (per vector)
1. Look up centroid values from indices
2. Inverse polar transform → get $\tilde{\mathbf{y}}$
3. $\tilde{\mathbf{x}} = \mathbf{S}^\top \cdot \tilde{\mathbf{y}}$ — undo preconditioning

In [ ]:
class PolarQuant:
    """Simplified PolarQuant implementation."""
    
    def __init__(self, d, n_levels=4, bits_per_level=None):
        self.d = d
        self.n_levels = n_levels  # how many recursion levels to quantize
        
        if bits_per_level is None:
            bits_per_level = [4] + [2] * (n_levels - 1)  # paper default
        self.bits_per_level = bits_per_level
        
        # Random preconditioning matrix (shared)
        G = np.random.randn(d, d)
        self.S, _ = np.linalg.qr(G)
        
        # Precompute codebooks per level
        self.codebooks = []
        self.boundary_list = []
        for level in range(1, n_levels + 1):
            b = bits_per_level[level - 1]
            n_codes = 2**b
            support = (0, 2*np.pi) if level == 1 else (0, np.pi/2)
            pdf_fn = lambda psi, l=level: angle_pdf(psi, l)
            centroids, boundaries = lloyd_max_1d(pdf_fn, support, n_codes, n_samples=30000)
            self.codebooks.append(centroids)
            self.boundary_list.append(boundaries)
    
    def _polar_transform_limited(self, x):
        """Polar transform for n_levels levels."""
        all_angles = []
        current = x.copy()
        for level in range(self.n_levels):
            n = len(current)
            if n < 2:
                break
            radii = np.zeros(n // 2)
            angles = np.zeros(n // 2)
            for j in range(n // 2):
                a, b = current[2*j], current[2*j + 1]
                radii[j] = np.sqrt(a**2 + b**2)
                if level == 0:
                    angles[j] = np.arctan2(b, a) % (2*np.pi)
                else:
                    angles[j] = np.arctan2(abs(b), abs(a))
            all_angles.append(angles)
            current = radii
        return current, all_angles  # remaining radii + angle vectors
    
    def _inverse_polar_limited(self, radii, all_angles):
        """Inverse polar transform."""
        current = radii.copy()
        for angles in reversed(all_angles):
            expanded = np.zeros(len(current) * 2)
            for j in range(len(current)):
                expanded[2*j] = current[j] * np.cos(angles[j])
                expanded[2*j + 1] = current[j] * np.sin(angles[j])
            current = expanded
        return current
    
    def quantize(self, x):
        """Quantize a vector."""
        # 1. Precondition
        y = self.S @ x
        
        # 2. Polar transform
        radii, angle_list = self._polar_transform_limited(y)
        
        # 3. Quantize angles
        quantized_indices = []
        for level, angles in enumerate(angle_list):
            boundaries = self.boundary_list[level]
            indices = np.searchsorted(boundaries[1:-1], angles)
            indices = np.clip(indices, 0, len(self.codebooks[level]) - 1)
            quantized_indices.append(indices)
        
        return radii, quantized_indices
    
    def dequantize(self, radii, quantized_indices):
        """Reconstruct vector from quantized representation."""
        # Look up centroids
        angle_list = []
        for level, indices in enumerate(quantized_indices):
            angles = self.codebooks[level][indices]
            angle_list.append(angles)
        
        # Inverse polar
        y_hat = self._inverse_polar_limited(radii, angle_list)
        
        # Undo preconditioning
        x_hat = self.S.T @ y_hat
        return x_hat
    
    def bits_per_coordinate(self):
        """Compute effective bits per coordinate."""
        total_bits = 0
        n_coords = self.d
        # Radius storage (remaining radii in full precision)
        n_radii = self.d // (2**self.n_levels)
        total_bits += n_radii * 16  # float16
        # Angles
        for level in range(self.n_levels):
            n_angles = self.d // (2**(level + 1))
            total_bits += n_angles * self.bits_per_level[level]
        return total_bits / self.d


# Test!
d = 128
pq = PolarQuant(d, n_levels=4, bits_per_level=[4, 2, 2, 2])

print(f"Effective bits per coordinate: {pq.bits_per_coordinate():.3f}")
print(f"Compression ratio: {16 / pq.bits_per_coordinate():.2f}x (vs float16)")
print()

# Test accuracy
n_trials = 500
mses = []
cosine_sims = []
for _ in range(n_trials):
    x = np.random.randn(d)
    radii, indices = pq.quantize(x)
    x_hat = pq.dequantize(radii, indices)
    
    mses.append(np.sum((x - x_hat)**2) / np.sum(x**2))  # relative MSE
    cosine_sims.append(np.dot(x, x_hat) / (np.linalg.norm(x) * np.linalg.norm(x_hat)))

print(f"Relative MSE:    {np.mean(mses):.6f} (lower is better)")
print(f"Cosine similarity: {np.mean(cosine_sims):.6f} (closer to 1.0 is better)")

---
## Step 6: Why No Normalization? — The Memory Saving

This is PolarQuant's **unique advantage** over methods like KIVI, GPTQ, etc.

### Traditional quantization
```
For each block of 128 values:
  1. Compute min, max → store as float16 (32 bits overhead)
  2. Scale to [0, 2^b - 1]
  3. Round to integer
  
Effective bits = b + 32/128 = b + 0.25
```

### PolarQuant
```
For each vector:
  1. Random precondition (shared matrix, no per-vector cost)
  2. Polar transform
  3. Angles are ALREADY in a known range [0, π/2]
  4. Quantize with precomputed codebook (no per-block stats!)
  
Effective bits = b + radius_overhead (just one float16 per vector)
```

The known distribution means angles don't need per-block min/max/scale/zero-point.  
One precomputed codebook works for **all vectors, all layers, all heads**.

In [ ]:
# Visualize memory layout comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Traditional: block of 128 values at 3 bits
ax = axes[0]
block_size = 128
data_bits = block_size * 3
overhead_bits = 32  # scale + zero_point in float16
total_trad = data_bits + overhead_bits
eff_trad = total_trad / block_size

ax.barh(['Overhead\n(scale+zero)'], [overhead_bits], color='#e74c3c', alpha=0.7, height=0.4)
ax.barh(['Data (3-bit)'], [data_bits], color='#3498db', alpha=0.7, height=0.4)
ax.barh(['Total'], [total_trad], color='#9b59b6', alpha=0.7, height=0.4)
ax.set_xlabel('Bits per block of 128')
ax.set_title(f'Traditional: {eff_trad:.2f} bits/value effective')
for i, v in enumerate([overhead_bits, data_bits, total_trad]):
    ax.text(v + 2, i, f'{v}', va='center', fontsize=11)

# PolarQuant: 4 levels
ax = axes[1]
d = 128
radius_bits = (d // 16) * 16  # 8 radii × 16 bits
level1_bits = (d // 2) * 4    # 64 angles × 4 bits
level2_bits = (d // 4) * 2    # 32 angles × 2 bits
level3_bits = (d // 8) * 2    # 16 angles × 2 bits
level4_bits = (d // 16) * 2   # 8 angles × 2 bits
total_polar = radius_bits + level1_bits + level2_bits + level3_bits + level4_bits
eff_polar = total_polar / d

labels = ['Radii (fp16)', 'Level 1 (4-bit)', 'Level 2 (2-bit)', 'Level 3 (2-bit)', 'Level 4 (2-bit)', 'Total']
values = [radius_bits, level1_bits, level2_bits, level3_bits, level4_bits, total_polar]
colors_bar = ['#f39c12', '#3498db', '#2ecc71', '#1abc9c', '#16a085', '#9b59b6']

ax.barh(labels, values, color=colors_bar, alpha=0.7, height=0.5)
ax.set_xlabel('Bits per block of 128')
ax.set_title(f'PolarQuant: {eff_polar:.3f} bits/value effective\n(NO per-block normalization overhead!)')
for i, v in enumerate(values):
    ax.text(v + 2, i, f'{v}', va='center', fontsize=10)

plt.suptitle('Memory layout comparison for d=128 vector', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f"Traditional 3-bit: {eff_trad:.2f} effective bits/value → {16/eff_trad:.2f}x compression")
print(f"PolarQuant:        {eff_polar:.3f} effective bits/value → {16/eff_polar:.2f}x compression")

---
## Step 7: Error Bound — The Theory

PolarQuant's main theorem states:

> For a $d$-dimensional Gaussian vector $\mathbf{x}$, using $O(\log(1/\varepsilon))$ bits per coordinate,  
> the reconstructed vector $\mathbf{x}'$ satisfies:
>
> $$\mathbb{E}\left[\|\mathbf{x} - \mathbf{x}'\|^2\right] = \varepsilon \cdot \|\mathbf{x}\|^2$$

The error is **proportional to** the vector's squared norm, scaled by $\varepsilon$ which decreases  
exponentially with the number of bits.

### Why does the error decrease at higher levels?

The total reconstruction error decomposes recursively:

$$\text{error}_0 \leq \frac{4d}{\alpha} \cdot \text{quant}_1 + \frac{4d}{\alpha}(1+\alpha) \cdot \text{quant}_2 + \ldots$$

where $\text{quant}_\ell$ is the angle quantization error at level $\ell$.  
Since $\text{Var}(\theta_\ell) = O(1/2^{\ell-1})$, the quantization error **shrinks exponentially**  
at higher levels, even with the same number of bits!

In [ ]:
# Measure reconstruction error vs dimension and bit allocation
dimensions = [16, 32, 64, 128, 256]
n_trials = 300

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) Relative MSE vs dimension (fixed bits)
ax = axes[0]
for bits_config, label, color in [
    ([4, 2, 2, 2], '4+2+2+2 bits', '#3498db'),
    ([3, 2, 2, 2], '3+2+2+2 bits', '#e74c3c'),
    ([2, 2, 2, 2], '2+2+2+2 bits', '#2ecc71'),
]:
    rel_mses = []
    for d in dimensions:
        if d < 2**len(bits_config):
            rel_mses.append(np.nan)
            continue
        pq = PolarQuant(d, n_levels=len(bits_config), bits_per_level=bits_config)
        trial_mses = []
        for _ in range(n_trials):
            x = np.random.randn(d)
            radii, indices = pq.quantize(x)
            x_hat = pq.dequantize(radii, indices)
            trial_mses.append(np.sum((x - x_hat)**2) / np.sum(x**2))
        rel_mses.append(np.mean(trial_mses))
    ax.plot(dimensions, rel_mses, '-o', label=label, color=color, lw=2, markersize=7)

ax.set_xlabel('Dimension (d)')
ax.set_ylabel('Relative MSE')
ax.set_title('Error vs Dimension')
ax.legend()
ax.grid(True, alpha=0.3)

# (b) Per-level angle quantization error
ax = axes[1]
d = 128
n_trials_level = 500
level_errors = {}

for b in [2, 3, 4]:
    pq_test = PolarQuant(d, n_levels=4, bits_per_level=[b]*4)
    errors_per_level = [[] for _ in range(4)]
    
    for _ in range(n_trials_level):
        x = np.random.randn(d)
        y = pq_test.S @ x
        _, angle_list = pq_test._polar_transform_limited(y)
        
        for level, angles in enumerate(angle_list):
            boundaries = pq_test.boundary_list[level]
            indices = np.searchsorted(boundaries[1:-1], angles)
            indices = np.clip(indices, 0, len(pq_test.codebooks[level]) - 1)
            quantized = pq_test.codebooks[level][indices]
            err = np.mean((angles - quantized)**2)
            errors_per_level[level].append(err)
    
    means = [np.mean(e) for e in errors_per_level]
    ax.plot([1, 2, 3, 4], means, '-o', label=f'{b}-bit per level', lw=2, markersize=7)

ax.set_xlabel('Level')
ax.set_ylabel('Angle quantization error')
ax.set_title('Higher levels → less error\n(angles are more concentrated)')
ax.set_yscale('log')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks([1, 2, 3, 4])

plt.tight_layout()
plt.show()

---
## Step 8: Inner Product Preservation — Critical for Attention

In a transformer, attention scores are computed as:

$$\text{score} = \frac{\mathbf{q} \cdot \mathbf{k}}{\sqrt{d}}$$

So PolarQuant must preserve **inner products** between query and key vectors.

The random preconditioning matrix $\mathbf{S}$ is **orthogonal**, meaning:

$$\langle \mathbf{S x}, \mathbf{S y} \rangle = \langle \mathbf{x}, \mathbf{y} \rangle$$

Inner products are perfectly preserved by preconditioning.  
The only source of error is the angle quantization itself.

In [ ]:
# Demonstrate inner product preservation
d = 128
pq = PolarQuant(d, n_levels=4, bits_per_level=[4, 2, 2, 2])

n_trials = 1000
true_ips = []
quant_ips = []

# Fixed "query" vector
q = np.random.randn(d)

for _ in range(n_trials):
    # Random "key" vector
    k = np.random.randn(d)
    
    # True inner product
    true_ip = np.dot(q, k)
    
    # Quantize key, compute inner product with original query
    radii, indices = pq.quantize(k)
    k_hat = pq.dequantize(radii, indices)
    quant_ip = np.dot(q, k_hat)
    
    true_ips.append(true_ip)
    quant_ips.append(quant_ip)

true_ips = np.array(true_ips)
quant_ips = np.array(quant_ips)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter: true vs quantized inner products
ax = axes[0]
ax.scatter(true_ips, quant_ips, alpha=0.2, s=10, color='steelblue')
lim = max(abs(true_ips).max(), abs(quant_ips).max()) * 1.05
ax.plot([-lim, lim], [-lim, lim], 'r--', lw=2, label='Perfect')
coeffs = np.polyfit(true_ips, quant_ips, 1)
ax.plot([-lim, lim], [coeffs[0]*(-lim)+coeffs[1], coeffs[0]*lim+coeffs[1]],
        'orange', lw=2, label=f'Fit: slope={coeffs[0]:.4f}')
ax.set_xlabel('True ⟨q, k⟩')
ax.set_ylabel('Quantized ⟨q, k̃⟩')
ax.set_title('Inner product preservation')
ax.legend()
ax.set_aspect('equal')
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)

# Error distribution
ax = axes[1]
errors = quant_ips - true_ips
ax.hist(errors, bins=50, density=True, alpha=0.7, color='steelblue', edgecolor='white')
ax.axvline(0, color='red', linestyle='--', lw=2)
ax.set_xlabel('Inner product error (quantized - true)')
ax.set_ylabel('Density')
ax.set_title(f'Error distribution\nMean={np.mean(errors):.4f}, Std={np.std(errors):.4f}')

plt.suptitle('PolarQuant preserves inner products with small, centered errors', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f"Correlation between true and quantized inner products: {np.corrcoef(true_ips, quant_ips)[0,1]:.6f}")

---
## Step 9: PolarQuant vs TurboQuant — Side by Side

Both methods come from overlapping research teams and share the philosophy of  
"rotate first, then quantize a predictable distribution." But they differ in key ways:

| | PolarQuant | TurboQuant |
|---|---|---|
| **What's quantized** | Angles in polar coords | Cartesian coordinates |
| **Transform** | Recursive polar ($\log_2 d$ levels) | Random rotation (one step) |
| **Distribution** | $\sin^{k}(2\psi)$ on angles | Beta → Gaussian on coords |
| **Normalization** | None needed (angles bounded) | None needed (known distribution) |
| **Unbiased IP** | Not explicitly | Yes (via QJL residual) |
| **Primary use** | KV cache in LLM inference | KV cache + vector databases |
| **Compression** | >4.2× | >4.5× |
| **Speed** | 14% faster generation than KIVI | Near-zero indexing time |

In [ ]:
# Compare PolarQuant vs simple uniform quantization (simulating traditional methods)
d = 128
n_trials = 500

def uniform_quantize(x, bits, block_size=32):
    """Traditional per-block uniform quantization."""
    x_q = np.zeros_like(x)
    for i in range(0, len(x), block_size):
        block = x[i:i+block_size]
        vmin, vmax = block.min(), block.max()
        if vmax - vmin < 1e-10:
            x_q[i:i+block_size] = block
            continue
        # Quantize
        n_levels = 2**bits
        scaled = (block - vmin) / (vmax - vmin) * (n_levels - 1)
        rounded = np.round(scaled)
        # Dequantize
        x_q[i:i+block_size] = rounded / (n_levels - 1) * (vmax - vmin) + vmin
    return x_q

# Run comparison
results = {'PolarQuant (3.9 bits)': [], 'Uniform 4-bit': [], 'Uniform 3-bit': [], 'Uniform 2-bit': []}

pq = PolarQuant(d, n_levels=4, bits_per_level=[4, 2, 2, 2])
q = np.random.randn(d)

for _ in range(n_trials):
    k = np.random.randn(d)
    true_ip = np.dot(q, k)
    
    # PolarQuant
    radii, indices = pq.quantize(k)
    k_hat = pq.dequantize(radii, indices)
    results['PolarQuant (3.9 bits)'].append(abs(np.dot(q, k_hat) - true_ip))
    
    # Uniform quantization at different bit widths
    for bits, name in [(4, 'Uniform 4-bit'), (3, 'Uniform 3-bit'), (2, 'Uniform 2-bit')]:
        k_unif = uniform_quantize(k, bits)
        results[name].append(abs(np.dot(q, k_unif) - true_ip))

# Plot comparison
fig, ax = plt.subplots(figsize=(10, 5))
methods = list(results.keys())
means = [np.mean(v) for v in results.values()]
colors = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c']

bars = ax.bar(methods, means, color=colors, alpha=0.8, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.4f}', ha='center', fontsize=11)

ax.set_ylabel('Mean absolute inner product error')
ax.set_title('PolarQuant vs Traditional Uniform Quantization\n(lower is better)')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
## Summary: The PolarQuant Algorithm

```
┌──────────────────────────────────────────────────────────┐
│                      SETUP (once)                        │
│  1. Generate random orthogonal matrix S                  │
│  2. Precompute angle codebooks per level via Lloyd-Max   │
└──────────────────────────────────────────────────────────┘

┌──────────────────────────────────────────────────────────┐
│              QUANTIZE (per KV vector)                     │
│  1. y = S · x                   (precondition)           │
│  2. (R, ψ¹, ψ², ...) = Polar(y) (recursive transform)  │
│  3. idx = nearest(ψ, codebook)   (quantize angles)      │
│  4. Store: R (fp16) + indices (2-4 bits each)            │
└──────────────────────────────────────────────────────────┘

┌──────────────────────────────────────────────────────────┐
│              DEQUANTIZE (per KV vector)                   │
│  1. ψ̃ = codebook[idx]           (look up angles)        │
│  2. ỹ = InversePolar(R, ψ̃)      (reconstruct in y-space)│
│  3. x̃ = Sᵀ · ỹ                 (undo preconditioning)  │
└──────────────────────────────────────────────────────────┘
```

### Key Takeaways:

1. **Polar coordinates** split a vector into angles + radius, enabling efficient compression
2. **Random preconditioning** makes angle distributions universal and analytically known
3. **No normalization overhead** — angles have known bounded ranges after preconditioning
4. **Higher recursion levels** have increasingly concentrated angles → need fewer bits
5. **>4.2× compression** of KV cache with near-lossless quality on long-context tasks